# RB-PLPE — Online Courses Dataset Cleaning
**Platforms:** Coursera · Udacity · Simplilearn · FutureLearn  
**Input:** `data/raw/Online_Courses.csv`  
**Output:** `data/cleaned/courses_cleaned.csv`  

## Imports & Paths

In [1]:
import pandas as pd
import numpy as np
import re
import os

RAW_PATH   = r"H:\resume_project\RB-PLPE\data\raw\Online_Courses.csv"
CLEAN_PATH = r"H:\resume_project\RB-PLPE\data\cleaned\courses_cleaned.csv"



os.makedirs(r"H:\resume_project\RB-PLPE\data\cleaned", exist_ok=True)

print("Input :", RAW_PATH)
print("Output:", CLEAN_PATH)

Input : H:\resume_project\RB-PLPE\data\raw\Online_Courses.csv
Output: H:\resume_project\RB-PLPE\data\cleaned\courses_cleaned.csv


## Load & Inspect Raw Dataset

In [2]:
df = pd.read_csv(RAW_PATH)

print("Shape   :", df.shape)
print("Columns :", df.columns.tolist())
print("\nPlatforms:")
print(df["Site"].value_counts())
print("\nCategories:")
print(df["Category"].value_counts().head(15))
print("\nNull counts:")
print(df.isnull().sum())
df.head(3)


Shape   : (8092, 45)
Columns : ['Unnamed: 0', 'Title', 'URL', 'Short Intro', 'Category', 'Sub-Category', 'Course Type', 'Language', 'Subtitle Languages', 'Skills', 'Instructors', 'Rating', 'Number of viewers', 'Duration', 'Site', 'Program Type', 'Courses', 'Level', 'Number of Reviews', 'Unique Projects', 'Prequisites', 'What you learn', 'Related Programs', 'Monthly access', '6-Month access', '4-Month access', '3-Month access', '5-Month access', '2-Month access', 'School', 'Topics related to CRM', 'ExpertTracks', 'FAQs', 'Course Title', 'Course URL', 'Course Short Intro', 'Weekly study', 'Premium course', "What's include", 'Rank', 'Created by', 'Program', 'Number of ratings', 'Price', 'COURSE CATEGORIES']

Platforms:
Site
Future Learn    4843
Coursera        2819
Udacity          282
Simplilearn      148
Name: count, dtype: int64

Categories:
Category
Business                            895
Computer Science                    455
Data Science                        448
Health           

,Unnamed: 0,Title,URL,Short Intro,Category,Sub-Category,Course Type,Language,Subtitle Languages,Skills,...,Course Short Intro,Weekly study,Premium course,What's include,Rank,Created by,Program,Number of ratings,Price,COURSE CATEGORIES
0,0,Machine Learning Specialization,https://www.coursera.org/specializations/machi...,#BreakIntoAI with Machine Learning Specializat...,Data Science,Machine Learning,Specialization,English,Subtitles: English,"Decision Trees, Artificial Neural Network, Log...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Introduction to Data Science Specialization,https://www.coursera.org/specializations/intro...,Launch your career in data science. Gain found...,Data Science,Data Analysis,Specialization,English,"Subtitles: English, Arabic, French, Portuguese...","Data Science, Relational Database Management S...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,Data Science Fundamentals with Python and SQL ...,https://www.coursera.org/specializations/data-...,Build the Foundation for your Data Science car...,Data Science,Data Analysis,Specialization,English,"Subtitles: English, Arabic, French, Portuguese...","Data Science, Github, Python Programming, Jupy...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Filter IT / CS / Data Science Categories

In [3]:
# Keep only IT relevant categories
# Remove Health, Arts, Social Sciences, Language Learning etc.
it_categories = [
    "Computer Science",
    "Data Science",
    "Information Technology",
    "Math and Logic"
]

In [4]:
before = len(df)

In [5]:
# Filter by category where available

df_cat = df[df["Category"].isin(it_categories)].copy()

In [6]:
# For rows with no category, check title for IT keywords
df_nocat = df[df['Category'].isna()].copy()

it_title_keywords = [
    "python", "java", "javascript", "sql", "data", "machine learning",
    "deep learning", "ai", "cloud", "devops", "cybersecurity", "web",
    "android", "ios", "react", "angular", "node", "aws", "azure",
    "software", "programming", "coding", "database", "network",
    "blockchain", "flutter", "docker", "kubernetes"
]

In [7]:
pattern = "|".join(it_title_keywords)

df_nocat = df_nocat[df_nocat["Title"].str.lower().str.contains(pattern, na=False)]


In [8]:
# Combine both
df = pd.concat([df_cat, df_nocat], ignore_index=True)

In [9]:
print(f"Before: {before}  →  After: {len(df)}")
print(f"Removed {before - len(df)} non-IT courses")


Before: 8092  →  After: 2566
Removed 5526 non-IT courses


## Keep Only Columns Needed for PPO Training

In [10]:
# Columns the PPO agent needs:
# course_title  → show to user
# skills        → match against missing skills
# difficulty    → sequence Beginner before Advanced
# rating        → reward signal for PPO
# course_url    → show to user
# duration_hrs  → estimate completion time
# platform      → show to user

df = df[[
    "Title",
    "Skills",
    "Level",
    "Rating",
    "URL",
    "Duration",
    "Site",
    "Category"
]].copy()

print(f"Shape: {df.shape}")
df.head(3)

Shape: (2566, 8)


,Title,Skills,Level,Rating,URL,Duration,Site,Category
0,Machine Learning Specialization,"Decision Trees, Artificial Neural Network, Log...",NaN,4.9stars,https://www.coursera.org/specializations/machi...,Approximately 3 months to complete,Coursera,Data Science
1,Introduction to Data Science Specialization,"Data Science, Relational Database Management S...",NaN,4.7stars,https://www.coursera.org/specializations/intro...,Approximately 5 months to complete,Coursera,Data Science
2,Data Science Fundamentals with Python and SQL ...,"Data Science, Github, Python Programming, Jupy...",NaN,4.6stars,https://www.coursera.org/specializations/data-...,Approximately 7 months to complete,Coursera,Data Science


## Clean Rating Column
> Raw value: `"4.9stars"` → Cleaned: `4.9`

In [11]:
def clean_rating(val):
    if pd.isna(val):
        return np.nan
    # Extract number from strings like "4.9stars" or "4.9"
    match = re.search(r"(\d+\.?\d*)", str(val))
    if match:
        return float(match.group(1))
    return np.nan


In [12]:
df['rating'] = df["Rating"].apply(clean_rating)

In [13]:
print("Rating distribution:")
print(df["rating"].describe())
print(f"\nNull ratings: {df['rating'].isna().sum()}")

Rating distribution:
count    1129.000000
mean        4.622852
std         0.212446
min         3.300000
25%         4.500000
50%         4.700000
75%         4.800000
max         5.000000
Name: rating, dtype: float64

Null ratings: 1437


## Clean Duration Column
> Convert "Approximately 3 months to complete" → hours

In [14]:
def duration_to_hours(val):
    if pd.isna(val):
        return np.nan
    val = str(val).lower()

    # Pattern: "X months at Y hours a week"
    # e.g. "6 months at 10 hours a week"
    match = re.search(r"(\d+)\s*month.*?(\d+)\s*hour", val)
    if match:
        months = int(match.group(1))
        hrs_per_week = int(match.group(2))
        return round(months * 4 * hrs_per_week, 1)  # weeks * hrs

    # Pattern: "Approximately X months to complete"
    match = re.search(r"(\d+)\s*month", val)
    if match:
        months = int(match.group(1))
        return round(months * 4 * 5, 1)  # assume 5 hrs/week default

    # Pattern: "X weeks"
    match = re.search(r"(\d+)\s*week", val)
    if match:
        weeks = int(match.group(1))
        return round(weeks * 5, 1)

    # Pattern: "X hours"
    match = re.search(r"(\d+)\s*hour", val)
    if match:
        return float(match.group(1))

    return np.nan

In [15]:
df["duration_hrs"] = df["Duration"].apply(duration_to_hours)

In [16]:
print("Duration stats:")
print(df["duration_hrs"].describe())
print(f"\nNull durations: {df['duration_hrs'].isna().sum()}")

# Fill missing durations with median
median_dur = df["duration_hrs"].median()
df["duration_hrs"] = df["duration_hrs"].fillna(median_dur)
print(f"\nFilled nulls with median: {median_dur} hrs")

Duration stats:
count    2464.000000
mean       26.883523
std        32.877049
min         1.000000
25%        11.000000
50%        17.000000
75%        25.000000
max       480.000000
Name: duration_hrs, dtype: float64

Null durations: 102

Filled nulls with median: 17.0 hrs


## Clean Skills Column

In [17]:
# Skills are comma separated in this dataset
def clean_skills(val):
    if pd.isna(val):
        return np.nan
    skills = val.split(",")
    skills = [s.strip().lower() for s in skills if s.strip()]
    return ", ".join(skills)

In [18]:
df["skills"] = df["Skills"].apply(clean_skills)

In [19]:
# Synonym normalization
synonym_map = {
    "machine learning (ml)"       : "machine learning",
    "artificial intelligence (ai)": "artificial intelligence",
    "data visualization (dataviz)": "data visualization",
    "python programming"          : "python",
    "python language"             : "python",
    "sql programming"             : "sql",
    "javascript (js)"             : "javascript",
    "deep learning (dl)"          : "deep learning",
    "natural language processing (nlp)": "natural language processing",
    "computer vision (cv)"        : "computer vision",
}

for old, new in synonym_map.items():
    df["skills"] = df["skills"].str.replace(old, new, case=False, regex=False)

In [20]:
print(f"Rows with skills: {df['skills'].notna().sum()}")
print(f"Rows without skills: {df['skills'].isna().sum()}")
print("\nSample:")
print(df["skills"].dropna().head(3).tolist())

Rows with skills: 968
Rows without skills: 1598

Sample:
['decision trees, artificial neural network, logistic regression, recommender systems, linear regression, regularization to avoid overfitting, gradient descent, supervised learning, logistic regression for classification, xgboost, tensorflow, tree ensembles', 'data science, relational database management system (rdbms), cloud databases, python, sql, deep learning, machine learning, big data, data mining, github, jupyter notebooks, rstudio', 'data science, github, python, jupyter notebooks, rstudio, data analysis, pandas, numpy, probability and statistics, regression analysis, data visualization, statistical hypothesis testing']


## Clean Level / Difficulty Column

In [21]:
# Normalize existing level values
level_map = {
    "beginner"    : "Beginner",
    "intermediate": "Intermediate",
    "advanced"    : "Advanced",
}
df["difficulty"] = df["Level"].str.strip().str.lower().map(level_map)

In [22]:
# Fill missing levels using keywords in title
def infer_level(row):
    if pd.notna(row["difficulty"]):
        return row["difficulty"]
    title = str(row["Title"]).lower()
    if any(x in title for x in ["beginner", "introduction", "intro", "fundamentals", "basics", "getting started", "for beginners"]):
        return "Beginner"
    elif any(x in title for x in ["advanced", "expert", "mastery", "professional"]):
        return "Advanced"
    else:
        return "Intermediate"   # default

df["difficulty"] = df.apply(infer_level, axis=1)

In [23]:
print("Difficulty distribution:")
print(df["difficulty"].value_counts())

Difficulty distribution:
difficulty
Intermediate    1910
Beginner         499
Advanced         157
Name: count, dtype: int64


## Rename Columns & Drop Unwanted

In [24]:
df = df.rename(columns={
    "Title"   : "course_title",
    "URL"     : "course_url",
    "Site"    : "platform",
    "Category": "category",
})

# Keep only final columns
df = df[[
    "course_title",
    "skills",
    "difficulty",
    "rating",
    "course_url",
    "duration_hrs",
    "platform",
    "category"
]].copy()

print("Columns:", df.columns.tolist())
df.head(3)

Columns: ['course_title', 'skills', 'difficulty', 'rating', 'course_url', 'duration_hrs', 'platform', 'category']


,course_title,skills,difficulty,rating,course_url,duration_hrs,platform,category
0,Machine Learning Specialization,"decision trees, artificial neural network, log...",Intermediate,4.9,https://www.coursera.org/specializations/machi...,60.0,Coursera,Data Science
1,Introduction to Data Science Specialization,"data science, relational database management s...",Beginner,4.7,https://www.coursera.org/specializations/intro...,100.0,Coursera,Data Science
2,Data Science Fundamentals with Python and SQL ...,"data science, github, python, jupyter notebook...",Beginner,4.6,https://www.coursera.org/specializations/data-...,140.0,Coursera,Data Science


## Remove Bad Rows

In [25]:
before = len(df)

# Drop rows with no skills — useless for gap analysis
df = df.dropna(subset=["skills"])
print(f"After removing no-skill rows: {len(df)}  (removed {before - len(df)})")

After removing no-skill rows: 968  (removed 1598)


In [26]:
# Drop rows with no course title
before = len(df)
df = df.dropna(subset=["course_title"])
print(f"After removing no-title rows: {len(df)}  (removed {before - len(df)})")

After removing no-title rows: 968  (removed 0)


In [27]:
# Drop zero-rated or very low rated courses
before = len(df)
df = df[df["rating"] >= 3.5]
print(f"After removing low-rated rows: {len(df)}  (removed {before - len(df)})")


After removing low-rated rows: 895  (removed 73)


In [28]:
# Drop duplicates — keep highest rated
before = len(df)
df = df.sort_values("rating", ascending=False)
df = df.drop_duplicates(subset=["course_title"], keep="first")
print(f"After removing duplicates: {len(df)}  (removed {before - len(df)})")

After removing duplicates: 766  (removed 129)


In [29]:
df = df.reset_index(drop=True)

## Final Quality Check

In [30]:
print("=" * 55)
print("FINAL DATASET REPORT")
print("=" * 55)
print(f"Shape      : {df.shape}")
print(f"Null values: {df.isnull().sum().sum()}")

print("\nDifficulty split:")
print(df["difficulty"].value_counts())

print("\nPlatform split:")
print(df["platform"].value_counts())

print("\nCategory split:")
print(df["category"].value_counts())

print(f"\nRating range : {df['rating'].min()} – {df['rating'].max()}")
print(f"Duration range: {df['duration_hrs'].min()} – {df['duration_hrs'].max()} hrs")

print("\nSample rows:")
df.head(5)

FINAL DATASET REPORT
Shape      : (766, 8)
Null values: 25

Difficulty split:
difficulty
Intermediate    572
Beginner        131
Advanced         63
Name: count, dtype: int64

Platform split:
platform
Coursera    741
Udacity      25
Name: count, dtype: int64

Category split:
category
Data Science              305
Computer Science          292
Information Technology    133
Math and Logic             11
Name: count, dtype: int64

Rating range : 3.5 – 5.0
Duration range: 1.0 – 480.0 hrs

Sample rows:


,course_title,skills,difficulty,rating,course_url,duration_hrs,platform,category
0,Road to the CISO – Culminating Project Course,"cyber-security regulation, cybersecurity staff...",Intermediate,5.0,https://www.coursera.org/learn/road-to-ciso?sp...,18.0,Coursera,Information Technology
1,Regression Analysis: Simplify Complex Data Rel...,"predictive modelling, statistical analysis, py...",Intermediate,5.0,https://www.coursera.org/learn/regression-anal...,28.0,Coursera,Data Science
2,Google Business Intelligence Professional Cert...,"data modeling, business analysis, bigquery, sh...",Advanced,5.0,https://www.coursera.org/professional-certific...,80.0,Coursera,Data Science
3,Introduction to Cybersecurity Essentials,"information security (infosec), cyberattacks, ...",Beginner,4.9,https://www.coursera.org/learn/introduction-to...,14.0,Coursera,Information Technology
4,Improving Deep Neural Networks: Hyperparameter...,"tensorflow, deep learning, mathematical optimi...",Intermediate,4.9,https://www.coursera.org/learn/deep-neural-net...,23.0,Coursera,Data Science


## Save to data/cleaned/

In [31]:
df.to_csv(CLEAN_PATH, index=False)

print(f"Saved  : {CLEAN_PATH}")
print(f"Rows   : {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print("\nDone! Ready to use for PPO training.")


Saved  : H:\resume_project\RB-PLPE\data\cleaned\courses_cleaned.csv
Rows   : 766
Columns: ['course_title', 'skills', 'difficulty', 'rating', 'course_url', 'duration_hrs', 'platform', 'category']

Done! Ready to use for PPO training.


## Done!

**Output:** `data/cleaned/courses_cleaned.csv`

| Column | Used for |
|--------|----------|
| `course_title` | Shown to user in pathway |
| `skills` | Matched against missing skills |
| `difficulty` | PPO course sequencing |
| `rating` | PPO reward signal |
| `course_url` | Shown to user |
| `duration_hrs` | Estimate completion time |
| `platform` | Shown to user |
| `category` | Domain filtering |